In [ ]:
!pip install librosa

In [ ]:
!pip install tensorflow

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

In [ ]:
# Path to the dataset directories
data_dir = 'ICBHI_final_database'  # Update this path to your dataset location
wav_dir = os.path.join(data_dir, 'wav_files')
annotations_dir = os.path.join(data_dir, 'annotations')

# Function to match WAV file and annotation file based on their base names
def get_file_pairs(wav_dir, annotations_dir):
    file_pairs = []
    for wav_file in os.listdir(wav_dir):
        if wav_file.endswith('.wav'):
            base_name = os.path.splitext(wav_file)[0]
            annotation_file = f"{base_name}.txt"
            annotation_path = os.path.join(annotations_dir, annotation_file)
            if os.path.exists(annotation_path):
                wav_path = os.path.join(wav_dir, wav_file)
                file_pairs.append((wav_path, annotation_path))
    return file_pairs

# Retrieve the matched file pairs
file_pairs = get_file_pairs(wav_dir, annotations_dir)


In [ ]:
# Function to extract audio segment
def extract_audio_segment(filename, start, end):
    y, sr = librosa.load(filename, sr=None, offset=start, duration=(end - start))
    return y, sr

# Function to extract labels
def get_label(crackles, wheezes):
    if crackles == 1 and wheezes == 1:
        return 2  # Crackles and Wheezes
    elif crackles == 1:
        return 0  # Crackles
    elif wheezes == 1:
        return 1  # Wheezes
    else:
        return 3  # Normal


In [ ]:
# Function to generate spectrogram and pad/trim to a fixed size
def generate_spectrogram(y, sr, n_mels=128, fixed_shape=(128, 128)):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    S_DB = librosa.power_to_db(S, ref=np.max)
    
    # Pad or trim the spectrogram to the fixed size
    if S_DB.shape[1] < fixed_shape[1]:
        pad_width = fixed_shape[1] - S_DB.shape[1]
        S_DB = np.pad(S_DB, ((0, 0), (0, pad_width)), mode='constant')
    elif S_DB.shape[1] > fixed_shape[1]:
        S_DB = S_DB[:, :fixed_shape[1]]
    
    return S_DB

# Function to load and preprocess data with fixed size spectrograms
def load_and_preprocess_data(file_pairs, fixed_shape=(128, 128)):
    X, y = [], []

    for wav_file, annotation_file in tqdm(file_pairs):
        # Load the annotation file
        annotations = pd.read_csv(annotation_file, delimiter='\t', header=None)
        annotations.columns = ['start', 'end', 'crackles', 'wheezes']

        # Process each respiratory cycle in the annotation file
        for _, row in annotations.iterrows():
            start = row['start']
            end = row['end']
            crackles = row['crackles']
            wheezes = row['wheezes']

            # Extract audio segment
            y_segment, sr = extract_audio_segment(wav_file, start, end)

            # Generate mel-spectrogram with fixed size
            spectrogram = generate_spectrogram(y_segment, sr, fixed_shape=fixed_shape)
            
            # Append features and label
            X.append(spectrogram)
            y.append(get_label(crackles, wheezes))

    # Convert lists to numpy arrays
    X = np.array(X)
    y = np.array(y)

    # Reshape X to match the CNN input shape requirements
    X = X[..., np.newaxis]  # Add channel dimension

    return X, y

# Load and preprocess data with fixed shape spectrograms
fixed_shape = (128, 128)  # This is an example; you can adjust it based on your data
X, y = load_and_preprocess_data(file_pairs, fixed_shape=fixed_shape)


In [ ]:
# Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(X_train.shape[1], X_train.shape[2], 1)),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')  # 4 output classes: Crackles, Wheezes, Both, Normal
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))


In [ ]:
# Define the mapping of class indices to labels
class_labels = ['Crackles', 'Wheezes', 'Both', 'Normal']

# Make predictions on the test set
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test, axis=1)

# Display the actual and predicted labels for the first 10 samples
print("Sample | Actual Label | Predicted Label")
print("----------------------------------------")
for i in range(50):  # Adjust the range to see more or fewer samples
    actual_label = class_labels[actual_classes[i]]
    predicted_label = class_labels[predicted_classes[i]]
    print(f"{i + 1:>6} | {actual_label:>12} | {predicted_label:>14}")

# Optional: Display a summary of the overall accuracy
correct_predictions = np.sum(predicted_classes == actual_classes)
total_predictions = len(actual_classes)
accuracy = correct_predictions / total_predictions
print(f"\nOverall Accuracy: {accuracy:.4f}")

# Optional: Count occurrences of each predicted class for more detailed analysis
from collections import Counter

predicted_counts = Counter(predicted_classes)
print("\nPredicted class distribution:")
for label_idx, count in predicted_counts.items():
    print(f"{class_labels[label_idx]}: {count} samples")



In [ ]:
# Save the entire model
model.save('lung_sound_classifier.keras')

In [ ]:
# Load the model
from tensorflow.keras.models import load_model

loaded_model = load_model('lung_sound_classifier.keras')